Attempt 2: Switching to RandomForest, Tuning Hyperparameters, and Ensemble with Logistic Regression (Best Score)

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score

# 1. Load Data

train_df = pd.read_csv("train-20251006-234245.csv")
test_df = pd.read_csv("test.csv")

# 2. Feature Engineering

def feature_engineering(df):
    # Family size
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # Title from Name
    df["Title"] = df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)
    df["Title"] = df["Title"].replace(
        ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare'
    )
    df["Title"] = df["Title"].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
    title_map = {'Mr':0,'Miss':1,'Mrs':2,'Master':3,'Rare':4}
    df["Title"] = df["Title"].map(title_map).fillna(4)

    # Encode Sex
    df["Sex"] = df["Sex"].map({'male': 0, 'female': 1}).astype(int)

    # Encode Embarked
    df["Embarked"] = df["Embarked"].map({'C': 0, 'Q': 1, 'S': 2})
    df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)

    # Age + bins
    df["Age"].fillna(df["Age"].median(), inplace=True)
    df["AgeBin"] = pd.cut(df["Age"], bins=[0,12,18,35,60,80], labels=False)

    # Fare + bins
    df["Fare"].fillna(df["Fare"].median(), inplace=True)
    df["FareBin"] = pd.qcut(df["Fare"], 4, labels=False)

    # Cabin known or not
    df["HasCabin"] = df["Cabin"].notnull().astype(int)

    return df

train_df = feature_engineering(train_df)
test_df = feature_engineering(test_df)

# 3. Prepare Train & Test Data
drop_cols = ["PassengerId","Name","Ticket","Cabin","Survived"]
X = train_df.drop(drop_cols, axis=1)
y = train_df["Survived"]

test_passenger_ids = test_df["PassengerId"]
X_test_final = test_df.drop(["PassengerId","Name","Ticket","Cabin"], axis=1)

# 4. Model Training & Validation
clf = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_split=5,
    random_state=42, n_jobs=-1
)

# Cross-validation accuracy
cv_scores = cross_val_score(clf, X, y, cv=5, scoring="accuracy")
print("Cross-validation scores:", cv_scores)
print("CV mean accuracy:", cv_scores.mean())

# Optional: single validation split check
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
clf.fit(X_train, y_train)
y_val_pred = clf.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))

# 5. Train on Full Data and Predict Kaggle Test
clf.fit(X, y)
test_preds = clf.predict(X_test_final)

# Submission file
submission = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Survived": test_preds
})
submission.to_csv("submission2.csv", index=False)
print("✅ submission.csv file created!")

Cross-validation scores: [0.84916201 0.8258427  0.82022472 0.80337079 0.84269663]
CV mean accuracy: 0.8282593685267716
Validation Accuracy: 0.8268156424581006
✅ submission.csv file created!
